# 第06回：Boltzで複合体を予測する
酵素 MfnG（チロシンをメチル化する酵素、384残基）と二つの低分子（SAH、チロシン）を入力し、Boltz-2 で複合体の原子座標を生成します。最後に、実験構造 7UX8 と比べます。GPUで実行する場合は、20段と40段の結果を同じ画面で比較できます。入力は Boltz 公式の例と同じタンパク質と低分子で、MSA は配布したものを使います。

**操作**：Googleアカウントでログイン →「ランタイム」→「ランタイムのタイプを変更」→ **T4 GPU** →「すべてのセルを実行」。「このノートブックはGoogleが作成したものではありません」と出たら「このまま実行」を押します。ダウンロードして開き直す必要はありません。編集を残すなら「ドライブにコピーを保存」を使います。

演習は20分を目安とし、採点・提出はありません。初回はプログラム・重み・分子辞書のダウンロードがあり、回線とGPUの割り当てにより待ち時間が変わります。T4 GPUを確保できない場合は、セル1で「保存済み結果を観察」を選び、CPUで同じ構造を回転・比較できます。このモードは新しい予測を行いません。

**条件**：リサイクリング1回・20段・1候補・seed 42。操作確認のため反復を減らしています（Boltzの既定は3回・200段）。研究で予測精度を判定する標準条件ではありません。外部のMSA検索は行いません。座標生成の反復は分子の実時間の運動ではありません。


In [ ]:
#@title 1. 実行方法と作業フォルダを選ぶ
execution_mode = "GPUで新しく予測" #@param ["GPUで新しく予測", "保存済み結果を観察"]
from pathlib import Path
import subprocess, sys, time, json, os, shutil
from datetime import datetime
from types import SimpleNamespace
started = time.perf_counter()
work = Path('/content/bioinfo06')
work.mkdir(parents=True,exist_ok=True)
os.chdir(work)
p = (subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],text=True,capture_output=True)
     if shutil.which('nvidia-smi') else SimpleNamespace(returncode=1,stdout='CPU'))
if execution_mode == 'GPUで新しく予測' and p.returncode != 0:
    print('GPUがありません。保存済み結果の観察へ切り替えます。新しい推論は行いません。')
    execution_mode = '保存済み結果を観察'
print('実行方法:',execution_mode,'／計算機:',p.stdout.strip())


## 準備
GPUモードでは独立したPython環境へBoltzを入れます。保存済み結果の観察モードでは、構造表示と比較に使う小さなライブラリだけを入れます。


In [ ]:
#@title 2. 表示用ライブラリとBoltzを準備
subprocess.run([sys.executable,'-m','pip','install','--only-binary=:all:','py3Dmol==2.5.3','gemmi==0.7.5'],check=True)
if execution_mode == '保存済み結果を観察':
    py = sys.executable
    print('保存済み結果の観察モード。Boltz・学習済み重みはダウンロードしません。')
else:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'uv', 'py3Dmol==2.5.3'], check=True)
    venv = work / '.venv'
    py = str(venv / 'bin/python')
    if not Path(py).exists():
        subprocess.run(['uv', 'venv', '--python', '3.11', str(venv)], check=True)
    subprocess.run(['uv', 'pip', 'install', '--python', py,
                    'torch==2.7.1', 'boltz==2.2.1', 'torchmetrics==1.7.4',
                    'numpy==1.26.4', 'rdkit==2025.3.3'], check=True)
    subprocess.run([py, '-c', "import torch; from importlib.metadata import version; print('Boltz',version('boltz'),'PyTorch',torch.__version__); assert torch.cuda.is_available(); x=torch.ones(4,device='cuda'); print('GPU calculation:',(x+x).cpu().tolist())"], check=True)
    print('準備完了。経過秒:', round(time.perf_counter()-started))


## 入力の意味
- **A**：タンパク質 MfnG のアミノ酸配列。チロシンの酸素にメチル基を付ける酵素です。固定した多重配列アラインメント（MSA）を使います。
- **C**：SAH（S-アデノシルホモシステイン）。メチル基を渡した後の補因子です。CCDという化学成分辞書の識別子で指定します。
- **E**：チロシン（基質）。SMILESという化学構造の文字列表現で指定します。立体化学も指定します。

公式の例から重複した鎖を除き、タンパク質1本と低分子を1個ずつにした入力です。実際の結晶は2本の鎖の二量体です。


In [ ]:
#@title 3. 配布したYAMLとMSAを展開する
import base64, zlib, hashlib
inputs = work/'inputs'
inputs.mkdir(exist_ok=True)
yaml_text = "version: 1  # Optional, defaults to 1\nsequences:\n  - protein:\n      id: A\n      sequence: MVTPEGNVSLVDESLLVGVTDEDRAVRSAHQFYERLIGLWAPAVMEAAHELGVFAALAEAPADSGELARRLDCDARAMRVLLDALYAYDVIDRIHDTNGFRYLLSAEARECLLPGTLFSLVGKFMHDINVAWPAWRNLAEVVRHGARDTSGAESPNGIAQEDYESLVGGINFWAPPIVTTLSRKLRASGRSGDATASVLDVGCGTGLYSQLLLREFPRWTATGLDVERIATLANAQALRLGVEERFATRAGDFWRGGWGTGYDLVLFANIFHLQTPASAVRLMRHAAACLAPDGLVAVVDQIVDADREPKTPQDRFALLFAASMTNTGGGDAYTFQEYEEWFTAAGLQRIETLDTPMHRILLARRATEPSAVPEGQASENLYFQ\n      msa: inputs/complex.a3m\n  - ligand:\n      id: C\n      ccd: SAH\n  - ligand:\n      id: E\n      smiles: 'N[C@@H](Cc1ccc(O)cc1)C(=O)O'\n"
msa_bytes = zlib.decompress(base64.b64decode('...'))
(inputs/'complex.yaml').write_text(yaml_text)
(inputs/'complex.a3m').write_bytes(msa_bytes)
print(yaml_text)
print('MSA SHA256:', hashlib.sha256(msa_bytes).hexdigest())

# GPUがなくても、授業と同じ実予測を操作できるように同梱。
import io, zipfile
reference_blob = base64.b64decode('...')
assert hashlib.sha256(reference_blob).hexdigest() == 'f8aaec3a5cc3d7fc247001c7d310213dbcd13e0339fdfe019a12a5aa2158ac02'
reference = work/'reference_20steps'
reference.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(reference_blob)) as z:
    z.extractall(reference)
shutil.copy2(reference/'7UX8.cif',work/'7UX8.cif')


## ノイズから座標を生成
`sampling_steps`は1候補を作る反復数、`diffusion_samples`は生成する候補の数です。`recycling_steps`は条件の特徴表現を更新する反復で、拡散の反復とは別です。学習済みの重みを使い、ここでは学習し直しません。

初回は公式サイトから重みと分子辞書を取得します。ログの進行と最後の「出力確認完了」を確認してください。

In [ ]:
#@title 4. Boltzを実行
sampling_steps = 20 #@param {type:"integer"}
diffusion_samples = 1 #@param {type:"integer"}
seed = 42 #@param {type:"integer"}
assert 2 <= sampling_steps <= 200
assert 1 <= diffusion_samples <= 2
if execution_mode == '保存済み結果を観察':
    run = reference
    pred = run/'predictions/complex'
    cifs = sorted(pred.glob('complex_model_*.cif'))
    meta = json.loads((run/'reference.json').read_text())
    (run/'execution.json').write_text(json.dumps(meta,ensure_ascii=False,indent=2))
    print('保存済みの20段・seed 42の結果です。上の段数を変更しても新たな推論は行いません。')
else:
    run = work / ('run_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
    cmd = [str(venv/'bin/boltz'), 'predict', str(inputs/'complex.yaml'),
           '--out_dir', str(run), '--model', 'boltz2', '--recycling_steps', '1',
           '--sampling_steps', str(sampling_steps), '--diffusion_samples', str(diffusion_samples),
           '--max_parallel_samples', '1', '--seed', str(seed), '--no_kernels',
           '--num_workers', '0', '--write_full_pae', '--write_full_pde']
    run.mkdir()
    t0 = time.perf_counter()
    with (run/'run.log').open('w') as log:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end=''); log.write(line); log.flush()
        rc = proc.wait()
    assert rc == 0, 'ログを確認してください。'
    pred = run/'boltz_results_complex/predictions/complex'
    cifs = sorted(pred.glob('complex_model_*.cif'))
    assert len(cifs) == diffusion_samples, '期待したCIFがありません。ログの失敗数を確認してください。'
    for f in cifs:
        assert f.stat().st_size > 1000
    meta = {'command':cmd, 'prediction_seconds':time.perf_counter()-t0,
            'total_seconds':time.perf_counter()-started, 'gpu':p.stdout.strip(), 'mode':'GPUで新しく予測', 'sampling_steps':sampling_steps, 'seed':seed, 'diffusion_samples':diffusion_samples, 'recycling_steps':1}
    (run/'execution.json').write_text(json.dumps(meta, indent=2))
    (run/'requirements.txt').write_text(subprocess.check_output(['uv','pip','freeze','--python',py], text=True))
    print('出力確認完了:', len(cifs), '候補。推論・初回データ取得を含む秒:', round(meta['prediction_seconds']))


## 信頼度を読む
BoltzのJSONとNPZではpLDDTは **0〜1** です。以下のグラフは比較しやすいよう100倍します。PAE（Å）は距離そのものではなく、相対配置の予測誤差です。pLDDT・ipTMはモデル自身が予測した信頼度で、実験構造との実測誤差ではありません。結合の強さでもありません。

この入力はタンパク質が1鎖だけなので、`protein_iptm=0`はタンパク質間界面がない場合の値です。タンパク質の形が低信頼という意味ではありません。

横軸・縦軸はトークン番号です。タンパク質では残基、低分子では原子がトークンなので、全体を「残基番号」と読まないでください。橙の線が鎖の境界です（1〜384：タンパク質、385〜410：SAH、411〜423：チロシン）。C末端の尾部（368〜384番）の値を、本体と比べてください。

In [ ]:
#@title 5. 信頼度の数値とグラフ
import numpy as np
import matplotlib.pyplot as plt
model_index = 0 #@param {type:"integer"}
assert 0 <= model_index < len(cifs)
tag = f'complex_model_{model_index}'
conf = json.loads((pred/f'confidence_{tag}.json').read_text())
for key in ['confidence_score','complex_plddt','ptm','iptm','ligand_iptm','protein_iptm']:
    if key in conf: print(key, ':', round(conf[key],3))
chain_names = ['A','C(SAH)','E(Tyr)']
print('pair_chains_iptm（行：位置を評価する鎖、列：基準にする鎖）')
for a_key,row in conf['pair_chains_iptm'].items():
    print(' ',chain_names[int(a_key)],[round(v,2) for v in row.values()])
lddt_file = np.load(pred/f'plddt_{tag}.npz')
pae_file = np.load(pred/f'pae_{tag}.npz')
plddt = lddt_file['plddt'].squeeze()
pae = pae_file['pae'].squeeze()
assert np.isfinite(plddt).all() and np.isfinite(pae).all()
assert plddt.min() >= 0 and plddt.max() <= 1.001
# トークンの並び：タンパク質の残基、SAHの原子、チロシンの原子
ends = np.cumsum([384,26,13])
assert ends[-1] == len(plddt)
for name,lo,hi in zip(chain_names,[0,ends[0],ends[1]],ends):
    print('平均pLDDT',name,round(float(plddt[lo:hi].mean()),3))
fig, ax = plt.subplots(1,2,figsize=(13,4))
ax[0].plot(np.arange(1,len(plddt)+1),100*plddt)
ax[0].set(xlabel='Token index (1-based)',ylabel='Predicted lDDT (0–100)',ylim=(0,100))
im=ax[1].imshow(pae,origin='upper',vmin=0,vmax=30,extent=(.5,pae.shape[1]+.5,pae.shape[0]+.5,.5))
ax[1].set(xlabel='Evaluated token',ylabel='Reference token',title='PAE (angstrom)')
for e in ends[:-1]:
    ax[0].axvline(e+.5,color='tab:orange',lw=1); ax[1].axvline(e+.5,color='tab:orange',lw=1); ax[1].axhline(e+.5,color='tab:orange',lw=1)
fig.colorbar(im,ax=ax[1]);fig.tight_layout()
fig.savefig(run/'confidence.png',dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(8,4))
im=ax.imshow(pae[:384,384:],vmin=0,vmax=30,aspect='auto',cmap='Greens_r',origin='upper')
ax.axvline(25.5,color='tab:orange',lw=2)
ax.set_xticks([12.5,32],['SAH (26 atoms)','Tyr (13 atoms)'])
ax.set(ylabel='Reference: protein token (0-based)',title='Protein-to-ligand PAE: enlarged block')
fig.colorbar(im,ax=ax,label='Predicted error (angstrom)');fig.tight_layout();plt.show()


## 立体構造を観察
タンパク質Aを青いリボン、SAH（C）を橙、チロシン（E）を紫のスティックで示します。ドラッグで回転、スクロールで拡大できます。

低分子が置かれた位置と、タンパク質表面との関係を見てください。低分子が近くに見えても、結合が実験で確認されたことにはなりません。

In [ ]:
#@title 6. 複合体の3D表示
import py3Dmol
cif = pred/f'{tag}.cif'
# CIFの座標をBoltz側のgemmiで読み、有限な座標があるか確認する。
check = "import gemmi,sys,math; s=gemmi.read_structure(sys.argv[1]); xyz=[v for m in s for c in m for r in c for a in r for v in (a.pos.x,a.pos.y,a.pos.z)]; assert xyz and all(map(math.isfinite,xyz)); print('Finite atoms:',len(xyz)//3)"
subprocess.run([py,'-c',check,str(cif)],check=True)
view = py3Dmol.view(width=850,height=520)
view.addModel(cif.read_text(),'cif')
view.setStyle({'chain':'A'},{'cartoon':{'color':'steelblue'}})
view.setStyle({'chain':'C'},{'stick':{'color':'orange','radius':.22}})
view.setStyle({'chain':'E'},{'stick':{'color':'purple','radius':.22}})
view.setBackgroundColor('white');view.zoomTo();view.show()


## 実験構造と重ねて比べる
MfnG の結晶構造 [7UX8](https://www.rcsb.org/structure/7UX8) には、SAH とチロシンが入っています。タンパク質の Cα で重ね合わせて、**タンパク質と低分子の両方**を3Dで比べます。低分子の RMSD は、この重ね合わせのまま原子の位置のずれを測った値です。

まず「複合体全体」で折り畳みの一致を見て、次に「結合部位」に切り替えてこのセルだけ再実行してください。タンパク質は予測が青、実験が灰色。低分子は予測が橙（SAH）と紫（チロシン）、実験が緑です。ドラッグで向きを変え、重なりとずれを確かめましょう。

実験の鎖Bと、それに結合した低分子を使います。結晶全体の二量体との比較ではありません。実験で座標が得られていない残基は、実験側には表示されず、Cα の RMSD にも含めません。

7UX8 は2022年に公開された構造で、Boltz-2 の学習データの期限（2023年6月）より前のものです。学習に含まれている可能性があるため、未知の複合体に対する性能の評価ではありません。信頼度と実際のずれを見比べる練習として使います。


In [ ]:
#@title 7. 実験構造 7UX8 と比べる
#@markdown 7UX8 の鎖Bの Cα で重ね合わせ、タンパク質と低分子のずれを計算します（授業用に追加したセル）。
#@markdown 全体の重なりを見たら、表示を「結合部位」に変えてこのセルだけ再実行してください。予測の再計算は不要です。
view_focus = "複合体全体" #@param ["複合体全体", "結合部位"]
import urllib.request
import py3Dmol
from IPython.display import HTML, display
script = r"""
import itertools, json, sys
import gemmi
import numpy as np
pred_file, exp_file, out_file = sys.argv[1:4]
pred = gemmi.read_structure(pred_file)[0]
exp = gemmi.read_structure(exp_file)[0]['B']
xyz = lambda a: np.array([a.pos.x, a.pos.y, a.pos.z])
ca_pred = {r.seqid.num: xyz(r['CA'][0]) for r in pred['A']}
ca_exp = {r.label_seq: xyz(r['CA'][0]) for r in exp.get_polymer() if r.find_atom('CA', '*')}
ids = sorted(set(ca_pred) & set(ca_exp))
E, P = np.array([ca_exp[i] for i in ids]), np.array([ca_pred[i] for i in ids])
# 実験構造を予測へ重ねる回転と並進（Kabsch法）
e0, p0 = E.mean(0), P.mean(0)
U, _, Vt = np.linalg.svd((E - e0).T @ (P - p0))
d = np.sign(np.linalg.det(Vt.T @ U.T))
R = Vt.T @ np.diag([1, 1, d]) @ U.T
t = p0 - R @ e0
rmsd = lambda A, B: float(np.sqrt(((A - B) ** 2).sum(1).mean()))
result = {'n_ca': len(ids), 'ca_rmsd': rmsd((R @ E.T).T + t, P)}
lig = {}
for r in exp:
    if r.name in ('SAH', 'TYR') and r.entity_type != gemmi.EntityType.Polymer:
        for a in r:
            if a.element.name != 'H' and (a.altloc in ('\0', 'A')):
                lig[(r.name, a.name)] = ((R @ xyz(a)) + t, a.element.name)
sah_pred = {a.name: xyz(a) for a in pred['C'][0]}
names = [n for n in sah_pred if ('SAH', n) in lig]
result['sah_rmsd'] = rmsd(np.array([lig[('SAH', n)][0] for n in names]), np.array([sah_pred[n] for n in names]))
result['sah_atoms'] = len(names)
# 予測のチロシンは SMILES の順に原子が並ぶ。環と末端のカルボキシ基の対称性を考えて最小の値を採る
order = ['N', 'CA', 'CB', 'CG', 'CD1', 'CE1', 'CZ', 'OH', 'CE2', 'CD2', 'C', 'O', 'OXT']
tyr_pred = np.array([xyz(a) for a in pred['E'][0] if a.element.name != 'H'])
best = None
for ring, carb in itertools.product((False, True), repeat=2):
    swap = {}
    if ring:
        swap.update({'CD1': 'CD2', 'CD2': 'CD1', 'CE1': 'CE2', 'CE2': 'CE1'})
    if carb:
        swap.update({'O': 'OXT', 'OXT': 'O'})
    v = rmsd(np.array([lig[('TYR', swap.get(n, n))][0] for n in order]), tyr_pred)
    best = v if best is None else min(best, v)
result['tyr_rmsd'] = best
# 数値計算と同じ回転・並進で、実験のタンパク質と低分子を一緒に移す。
# 結晶水・他の化合物は表示しない。残基番号とATOM/HETATMの区別は保持する。
aligned_structure = gemmi.Structure()
aligned_model = gemmi.Model('1')
aligned_chain = gemmi.Chain('B')
for residue in exp:
    if residue.entity_type != gemmi.EntityType.Polymer and residue.name not in ('SAH', 'TYR'):
        continue
    moved = residue.clone()
    # Cαの計算と同じく、各原子名の最初の配座を使う（A以外しかない残基も保持）。
    selected = {}
    for atom in moved:
        selected.setdefault(atom.name, atom.altloc)
    for k in range(len(moved) - 1, -1, -1):
        atom = moved[k]
        if atom.element.name == 'H' or atom.altloc != selected[atom.name]:
            del moved[k]
            continue
        pos = R @ xyz(atom) + t
        atom.pos = gemmi.Position(*pos)
        atom.altloc = '\0'
        atom.aniso = gemmi.SMat33f(0, 0, 0, 0, 0, 0)
    aligned_chain.add_residue(moved)
aligned_model.add_chain(aligned_chain)
aligned_structure.add_model(aligned_model)
aligned_structure.write_pdb(out_file)
print(json.dumps(result))
"""
try:
    exp_file = work/'7UX8.cif'
    if not exp_file.exists():
        urllib.request.urlretrieve('https://files.rcsb.org/download/7UX8.cif', exp_file)
    (work/'compare.py').write_text(script)
    aligned = run/f'7UX8_complex_aligned_{tag}.pdb'
    out = subprocess.run([py, str(work/'compare.py'), str(cif), str(exp_file), str(aligned)], text=True, capture_output=True)
    assert out.returncode == 0, out.stderr[-500:]
    res = json.loads(out.stdout.strip().splitlines()[-1])
    (run/'comparison_7UX8.json').write_text(json.dumps(res, indent=2))
    print(f"タンパク質の Cα  {res['n_ca']:3d} 残基  RMSD {res['ca_rmsd']:.2f} Å")
    print(f"SAH             {res['sah_atoms']:3d} 原子  RMSD {res['sah_rmsd']:.2f} Å")
    print(f"チロシン         13 原子  RMSD {res['tyr_rmsd']:.2f} Å")
    display(HTML('<p><b>タンパク質</b>：予測＝青、実験＝灰色　'
                 '<b>低分子</b>：予測SAH＝橙、予測チロシン＝紫、実験＝緑</p>'
                 '<p>ドラッグで回転、スクロールで拡大・縮小できます。'
                 'タンパク質のCαで重ねた後、低分子は個別に合わせ直していません。</p>'))
    view = py3Dmol.view(width=850, height=600)
    view.addModel(cif.read_text(), 'cif')
    view.addModel(aligned.read_text(), 'pdb')
    view.setStyle({'model':0,'chain':'A'}, {'cartoon':{'color':'#2878B5'}})
    view.setStyle({'model':0,'chain':'C'}, {'stick':{'color':'orange','radius':.22}})
    view.setStyle({'model':0,'chain':'E'}, {'stick':{'color':'purple','radius':.22}})
    view.setStyle({'model':1,'hetflag':False}, {'cartoon':{'color':'#A8A8A8','opacity':.75}})
    view.setStyle({'model':1,'hetflag':True}, {'stick':{'color':'#249748','radius':.14}})
    view.setBackgroundColor('white')
    if view_focus == '結合部位':
        # 低分子を隠さないように、拡大表示ではタンパク質を薄くする。
        view.setStyle({'model':0,'chain':'A'}, {'cartoon':{'color':'#2878B5','opacity':.2}})
        view.setStyle({'model':1,'hetflag':False}, {'cartoon':{'color':'#A8A8A8','opacity':.15}})
        view.zoomTo({'model':0,'chain':['C','E']})
        view.zoom(.7)
    else:
        view.zoomTo()
    view.show()
except Exception as error:
    print('比較を表示できませんでした。上のセルまで実行済みか確認してください：', error)
    raise

# 成功した比較の実測値と条件を、このランタイムに保存する。
record = dict(meta, model=tag, run=str(run), cif=str(cif), comparison=res,
              confidence={k:float(conf[k]) for k in ('complex_plddt','iptm','ligand_iptm') if k in conf})
(run/f'comparison_record_{tag}.json').write_text(json.dumps(record,ensure_ascii=False,indent=2))


## 計算条件を変えて結果を比較
セル4で段数を20から40へ変え、セル4〜7を再実行します。下のセルは成功した比較の数値を保存履歴から読み、最新2件を同じ実験構造に重ねて表示します。モデルの番号は順位とは限りません。

段数を増やせばこの1例のRMSDが必ず下がるわけではありません。信頼度と実測誤差を別々に読みます。保存済み結果モードでは、元の20段の結果だけを表示します。

In [ ]:
#@title 8. 計算条件・実験とのずれ・構造を並べて比較
from IPython.display import display, HTML
import html
records=[]
for f in sorted(work.glob('*/comparison_record_*.json')):
    r=json.loads(f.read_text())
    if Path(r['cif']).exists(): records.append(r)
assert records, '先にセル7を実行してください。'
rows=['<table style="border-collapse:collapse"><tr><th>結果</th><th>段数</th><th>seed</th><th>時間（秒）</th><th>pLDDT（0–100）</th><th>ipTM</th><th>Cα RMSD（Å）</th><th>SAH RMSD（Å）</th><th>Tyr RMSD（Å）</th></tr>']
for i,r in enumerate(records):
    c=r['comparison'];q=r['confidence'];label=r.get('mode','計算')
    values=[f"{i}: {label}",r.get('sampling_steps','?'),r.get('seed','?'),round(r['prediction_seconds'],1) if 'prediction_seconds' in r else '記録なし',f"{100*q['complex_plddt']:.1f}",f"{q['iptm']:.3f}",f"{c['ca_rmsd']:.3f}",f"{c['sah_rmsd']:.3f}",f"{c['tyr_rmsd']:.3f}"]
    rows.append('<tr>'+''.join('<td style="border:1px solid #aaa;padding:8px">'+html.escape(str(v))+'</td>' for v in values)+'</tr>')
display(HTML(''.join(rows)+'</table>'))
print('20段と40段を比べる場合はGPUモードで段数だけ変更し、セル4〜7を再実行します。')
print('新しい2件を表示。同じseed・候補数・MSAなどの条件か、表と実行記録を確認してください。')
selected=records[-2:]
# Each prediction is aligned to the same experimental protein reference, never to its own ligand.
align_script = r"""
import sys,gemmi,numpy as np
s=gemmi.read_structure(sys.argv[1]);e=gemmi.read_structure(sys.argv[2])[0]['B']
xyz=lambda a:np.array([a.pos.x,a.pos.y,a.pos.z])
P={r.seqid.num:xyz(r['CA'][0]) for r in s[0]['A']}
E={r.label_seq:xyz(r['CA'][0]) for r in e.get_polymer() if r.find_atom('CA','*')}
ids=sorted(set(P)&set(E));p=np.array([P[i] for i in ids]);q=np.array([E[i] for i in ids]);a=p.mean(0);b=q.mean(0)
u,_,v=np.linalg.svd((p-a).T@(q-b));R=v.T@np.diag([1,1,np.sign(np.linalg.det(v.T@u.T))])@u.T
for model in s:
 for chain in model:
  for residue in chain:
   for atom in residue:
    atom.pos=gemmi.Position(*(R@(xyz(atom)-a)+b));atom.aniso=gemmi.SMat33f(0,0,0,0,0,0)
s.write_pdb(sys.argv[3])
ref=gemmi.Structure();m=gemmi.Model('1');m.add_chain(e.clone());ref.add_model(m);ref.remove_waters();ref.write_pdb(sys.argv[4])
"""
(work/'align_for_comparison.py').write_text(align_script)
viewer=py3Dmol.view(width=900,height=480,viewergrid=(1,len(selected)),linked=True)
for j,r in enumerate(selected):
    folder=Path(r['run']);aligned=folder/f"common_reference_{r['model']}.pdb";exp_pdb=work/'reference_B.pdb'
    subprocess.run([py,str(work/'align_for_comparison.py'),r['cif'],str(work/'7UX8.cif'),str(aligned),str(exp_pdb)],check=True)
    pos=(0,j)
    viewer.addModel(aligned.read_text(),'pdb',viewer=pos)
    viewer.addModel(exp_pdb.read_text(),'pdb',viewer=pos)
    viewer.setStyle({'model':0,'chain':'A'},{'cartoon':{'color':'#2878B5','opacity':.35}},viewer=pos)
    viewer.setStyle({'model':0,'chain':'C'},{'stick':{'color':'orange','radius':.22}},viewer=pos)
    viewer.setStyle({'model':0,'chain':'E'},{'stick':{'color':'purple','radius':.22}},viewer=pos)
    viewer.setStyle({'model':1,'hetflag':False},{'cartoon':{'color':'#A8A8A8','opacity':.15}},viewer=pos)
    viewer.setStyle({'model':1,'resn':['SAH','TYR'],'hetflag':True},{'stick':{'color':'#249748','radius':.14}},viewer=pos)
    viewer.zoomTo({'model':0,'chain':['C','E']},viewer=pos)
    viewer.zoom(.7,viewer=pos)
    viewer.addLabel(f"{r.get('sampling_steps','?')} steps / seed {r.get('seed','?')}",{'useScreen':True,'position':{'x':10,'y':10},'backgroundColor':'white','fontColor':'black','fontSize':15},viewer=pos)
viewer.setBackgroundColor('white');viewer.show()
print('予測：タンパク質＝青、SAH＝橙、Tyr＝紫。実験：タンパク質＝灰色、低分子＝緑。')
print('両画面を同じ実験タンパク質の座標へ重ねています。ドラッグで回転できます。')


## 結果を残す
ZIPには入力・MSA、予測CIF、信頼度、実行条件、ログ、パッケージ一覧を含めます。Colabのランタイムが消える前に保存してください。重みやインストール環境はZIPへ入れません。

余裕がある場合は`sampling_steps=40`に変え、セル4〜7を再実行します。出力先は実行ごとに分かれます。候補の違いと予測精度の改善は別です。信頼度だけで「正解に近づいた」とは判断しません。

In [ ]:
#@title 9. 結果ZIPを保存
import shutil
from google.colab import files
shutil.copytree(inputs,run/'inputs',dirs_exist_ok=True)
import zipfile
archive = str(work/'results.zip')
folders = {run} | {p.parent for p in work.glob('*/comparison_record_*.json')}
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as bundle:
    for folder in sorted(folders):
        for f in sorted(folder.rglob('*')):
            if f.is_file(): bundle.write(f,f.relative_to(work))
    for f in sorted(inputs.glob('*')):
        if f.is_file(): bundle.write(f,f.relative_to(work))
print('保存するファイル:', archive)
files.download(archive)


## 出典
- [Boltz公式](https://github.com/jwohlwend/boltz)・[予測の入出力](https://github.com/jwohlwend/boltz/blob/b1ebfc46ecf57f5414e0d1a6f9027bbb122c53bc/docs/prediction.md)
- [Boltz-2論文](https://doi.org/10.1101/2025.06.14.659707)
- 入力：公式examplesを基にしたLearning Resources「Boltzの理論と実装」の複合体例。BoltzコードはMITライセンス。配布MSAと入力の由来も同じ公式例です。
- [Colab_Boltz-2](https://github.com/JKourelis/Colab_Boltz-2)の実行手順を参考に、授業用の固定入力・独立環境・最小限のセル構成で作成しました。
